# Notebook 03: Model Development

Train and compare **Linear Regression**, **Random Forest**, and **Support Vector Regression (SVR)** on the cleaned country–year panel. Use a **time-based train/test split** to reduce future-information leakage.


## Load data and define train/test split


In [ ]:
from IPython.display import display
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path("../data/processed/country_year_modeling_panel_cleaned.csv")
df = pd.read_csv(DATA_PATH)

df["year"] = df["year"].astype(int)
if "log_mmr" not in df.columns:
    df["log_mmr"] = np.log1p(df["mmr"])

target = "log_mmr"
features = [
    "gdp_pc", "health_exp_gdp", "fertility", "skilled_births",
    "pm25", "heat_index35_days", "cooling_degree_days",
]
features = [c for c in features if c in df.columns]

split_year = 2018
train = df[df["year"] <= split_year]
test = df[df["year"] > split_year]
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]
print("Train:", X_train.shape, "Test:", X_test.shape)


## Feature scaling and why it matters

**Support Vector Regression** is sensitive to the **scale** of inputs because it relies on distances and margins in feature space. **Linear Regression** with regularization or gradient-based fitting also benefits from comparable feature scales.

**Random Forest** is tree-based and typically **does not require** scaling; trees split on ordered feature values, so we use imputation only in that branch.

Below, `StandardScaler` is applied inside `Pipeline` steps for **Linear Regression** and **SVR**, not for **Random Forest**.


In [ ]:
# Pipelines: StandardScaler for Linear Regression and SVR; not for Random Forest
lin_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
])
svr_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", SVR(kernel="rbf", C=10.0, epsilon=0.1)),
])

models = {
    "Linear Regression": lin_pipe,
    "Random Forest": rf_pipe,
    "SVR (RBF)": svr_pipe,
}

rows = []
for name, est in models.items():
    est.fit(X_train, y_train)
    pred_log = est.predict(X_test)
    pred = np.expm1(pred_log)
    actual = np.expm1(y_test)
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(actual, pred),
        "RMSE": rmse,
        "R2_log": r2_score(y_test, pred_log),
    })

results = pd.DataFrame(rows).sort_values("RMSE")
display(results)
